# 02 - Praproses dan Pembagian Data

Tahap ini melanjutkan hasil EDA (`notebooks/01_dataset_analysis.ipynb`) dan menjalankan penanganan temuan EDA sampai siap training. Isi: **cross-split leakage** (bagian 2), **duplikat dalam-split** (bagian 3), **manifest train bersih** (bagian 4), **bobot kelas** (bagian 5), **split train/validation stratified** (bagian 6), **pipeline resize + normalisasi** (bagian 7), dan **penggabungan dataset GCD+CCSN_v2** (bagian 8, opsional).

**Batasan.** Notebook hanya membaca manifest audit dan menulis daftar/manifest/stat ke `reports/tables/` serta ringkasan ke `reports/metrics/`. Berkas di `data/raw/` **tidak** diubah, dipindah, atau dihapus, dan `data/processed/` **tidak** ditulis — pipeline praproses bersifat di memori saat pemuatan.

**Kode yang dipakai ulang.** `scripts/build_leakage_exclusion.py` (`read_manifest`, `find_exclusions`, `find_within_split_duplicates`), `scripts/make_split.py` (`grouped_stratified_split`), dan `src/datasets/image_stats.py` (`channel_mean_std`). Konteks keputusan tiap langkah: `docs/dataset/DATASET_QUALITY_SUMMARY.md` §2.1–§2.5.

**Prasyarat.** `reports/tables/gcd_raw_manifest.csv` harus ada (dihasilkan notebook 01 / `scripts/audit_dataset.py`).

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from scripts.build_leakage_exclusion import (
    DEDUP_SPLITS, SPLIT_PRIORITY, find_exclusions, find_within_split_duplicates, read_manifest,
)

SEED = 42  # mengikuti configs/base.yaml
DATASET = "gcd_raw"
TABLE_DIR = PROJECT_DIR / "reports" / "tables"
MANIFEST = TABLE_DIR / f"{DATASET}_manifest.csv"
assert MANIFEST.exists(), f"Manifest audit tidak ada: {MANIFEST}. Jalankan notebook 01 atau scripts/audit_dataset.py dulu."

rows = read_manifest(MANIFEST)       # logika sama dengan scripts/build_leakage_exclusion.py
man = pd.read_csv(MANIFEST, dtype=str)
man["split"] = man["split"].str.lower()
man = man[man["readable"] == "1"].reset_index(drop=True)
print("Manifest     :", MANIFEST.relative_to(PROJECT_DIR).as_posix())
print("Citra terbaca:", f"{len(man):,}")
for s, n in man["split"].value_counts().items():
    print(f"  {s:5s}: {n:,}")

Manifest     : reports/tables/gcd_raw_manifest.csv
Citra terbaca: 19,000
  train: 10,000
  test : 9,000


## 2. Cross-split leakage

Dua sumber kebocoran lintas split ditangani, keduanya dengan **mempertahankan test** dan membuang salinan di **train**:

1. **Piksel identik** (`pixel_sha256` sama) — eksak, dicocokkan dengan tabel EDA `gcd_raw_eda_cross_split_leakage.csv`.
2. **Near-duplicate terkonfirmasi** — pasangan train↔test dengan jarak dHash ≤ 2 yang, lewat **tinjauan visual** (`reports/figures/eda/near_dup_xsplit_*.png`), terbukti citra yang sama (disimpan/dikompresi ulang sehingga piksel berbeda). Kandidat berjarak ≥ 3 sudah ditinjau dan **bukan** citra sama (false positive dHash pada langit berdetail rendah), jadi dipertahankan.

In [2]:
exclusions = find_exclusions(rows)   # grup piksel-identik lintas split; keep = prioritas tertinggi
excl = pd.DataFrame(exclusions)

print("Prioritas split (dipertahankan dari kiri):", SPLIT_PRIORITY)
print("Total citra dikeluarkan (leakage):", len(excl))
print()
print(excl.groupby(["split", "class_dir"]).size().rename("dikeluarkan").to_string())

# Cross-check dengan tabel EDA cross-split leakage (harus set yang sama)
leak = pd.read_csv(TABLE_DIR / f"{DATASET}_eda_cross_split_leakage.csv")
train_in_leak = {p for col in ("File A", "File B") for p in leak[col] if str(p).startswith("train/")}
assert set(excl["rel_path"]) == train_in_leak, "Daftar pengecualian tidak cocok dengan tabel EDA."
print(f"\nCross-check EDA: {len(train_in_leak)} citra train cocok dengan tabel cross-split leakage.")

EXCL_CSV = TABLE_DIR / f"{DATASET}_leakage_exclusion.csv"
excl.to_csv(EXCL_CSV, index=False, encoding="utf-8")
print("Ditulis:", EXCL_CSV.relative_to(PROJECT_DIR).as_posix())

# (2) Near-duplicate lintas split yang DIKONFIRMASI sama lewat tinjauan visual.
# Ambang d<=2 adalah hasil tinjauan (bagian ambigu d>=3 ditinjau dan bukan citra sama).
NEAR_DUP_CONFIRM_MAX_HAMMING = 2
nd = pd.read_csv(TABLE_DIR / f"{DATASET}_eda_near_duplicate_candidates.csv")
nd_x = nd[(nd["location"] == "train-test") & (nd["hamming"] <= NEAR_DUP_CONFIRM_MAX_HAMMING)]
near_excl = pd.DataFrame({
    "rel_path": nd_x["file_a"].to_numpy(),          # sisi train
    "split": "train",
    "class_dir": nd_x["class_a"].to_numpy(),
    "reason": "near_duplicate_cross_split",
    "match_type": "near_duplicate_dhash",
    "keep_split": "test",
    "keep_rel_path": nd_x["file_b"].to_numpy(),
    "hamming": nd_x["hamming"].to_numpy(),
})
NEAR_CSV = TABLE_DIR / f"{DATASET}_near_dup_crosssplit_exclusion.csv"
near_excl.to_csv(NEAR_CSV, index=False, encoding="utf-8")
print(f"\nNear-duplicate lintas split dikonfirmasi (d<={NEAR_DUP_CONFIRM_MAX_HAMMING}): {len(near_excl)} citra train dibuang.")
print(f"Kandidat d>={NEAR_DUP_CONFIRM_MAX_HAMMING + 1} ditinjau manual dan TIDAK dibuang (bukan citra sama).")
print("Ditulis:", NEAR_CSV.relative_to(PROJECT_DIR).as_posix())

Prioritas split (dipertahankan dari kiri): ['test', 'val', 'validation', 'train']
Total citra dikeluarkan (leakage): 156

split  class_dir 
train  4_clearsky    155
       7_mixed         1

Cross-check EDA: 156 citra train cocok dengan tabel cross-split leakage.
Ditulis: reports/tables/gcd_raw_leakage_exclusion.csv

Near-duplicate lintas split dikonfirmasi (d<=2): 32 citra train dibuang.
Kandidat d>=3 ditinjau manual dan TIDAK dibuang (bukan citra sama).
Ditulis: reports/tables/gcd_raw_near_dup_crosssplit_exclusion.csv


## 3. Duplikat dalam-split

Citra piksel-identik yang muncul lebih dari sekali **di dalam satu split**. Pada split yang dideduplikasi (`DEDUP_SPLITS` = train, val) satu representatif dipertahankan (rel_path terkecil) dan salinannya dikeluarkan. Pada **test** duplikat **tidak dikeluarkan** (test resmi utuh), hanya dicatat agar diketahui saat membaca metrik.

In [3]:
within_excl, within_kept = find_within_split_duplicates(rows)
dup_excl = pd.DataFrame(within_excl)
dup_kept = pd.DataFrame(within_kept)

DUP_CSV = TABLE_DIR / f"{DATASET}_within_split_duplicate_exclusion.csv"
KEPT_CSV = TABLE_DIR / f"{DATASET}_test_duplicates.csv"
dup_excl.to_csv(DUP_CSV, index=False, encoding="utf-8")
dup_kept.to_csv(KEPT_CSV, index=False, encoding="utf-8")

cols = ["rel_path", "keep_rel_path", "class_dir"]
print(f"Dedup split: {DEDUP_SPLITS}")
print(f"\nDuplikat dalam-split DIKELUARKAN (train/val): {len(dup_excl)}")
print(dup_excl[cols].to_string(index=False) if not dup_excl.empty else "  (tidak ada)")
print(f"\nDuplikat dalam-split DICATAT saja (test, tidak dikeluarkan): {len(dup_kept)}")
print(dup_kept[cols].to_string(index=False) if not dup_kept.empty else "  (tidak ada)")
print("\nDitulis:", DUP_CSV.relative_to(PROJECT_DIR).as_posix())
print("Ditulis:", KEPT_CSV.relative_to(PROJECT_DIR).as_posix())

Dedup split: ('train', 'val')

Duplikat dalam-split DIKELUARKAN (train/val): 1
                        rel_path                    keep_rel_path class_dir
train/7_mixed/7_mixed_000291.jpg train/7_mixed/7_mixed_000088.jpg   7_mixed

Duplikat dalam-split DICATAT saja (test, tidak dikeluarkan): 2
                                       rel_path                                   keep_rel_path       class_dir
            test/1_cumulus/1_cumulus_001420.jpg             test/1_cumulus/1_cumulus_001291.jpg       1_cumulus
test/5_stratocumulus/5_stratocumulus_003369.jpg test/5_stratocumulus/5_stratocumulus_003365.jpg 5_stratocumulus

Ditulis: reports/tables/gcd_raw_within_split_duplicate_exclusion.csv
Ditulis: reports/tables/gcd_raw_test_duplicates.csv


## 4. Manifest train bersih

Manifest train bersih = baris train pada manifest dikurangi gabungan citra yang dikeluarkan (leakage lintas split + duplikat dalam-split train). Diverifikasi: tidak ada lagi citra train yang piksel-identik dengan test, dan tidak ada lagi duplikat di dalam train.

In [4]:
excluded = set(excl["rel_path"])                 # leakage piksel-identik lintas split
if not dup_excl.empty:
    excluded |= set(dup_excl["rel_path"])        # duplikat dalam-split train
if not near_excl.empty:
    excluded |= set(near_excl["rel_path"])       # near-duplicate lintas split terkonfirmasi

train = man[man["split"] == "train"]
clean_train = train[~train["rel_path"].isin(excluded)].reset_index(drop=True)
CLEAN_CSV = TABLE_DIR / f"{DATASET}_train_clean_manifest.csv"
clean_train.to_csv(CLEAN_CSV, index=False, encoding="utf-8")

# Verifikasi
test_px = set(man.loc[man["split"] == "test", "pixel_sha256"])
remaining_leak = int(clean_train["pixel_sha256"].isin(test_px).sum())
dups_in_clean = int(clean_train["pixel_sha256"].duplicated().sum())
assert remaining_leak == 0, f"Masih ada {remaining_leak} citra train identik dengan test."
assert dups_in_clean == 0, f"Masih ada {dups_in_clean} duplikat di dalam train bersih."

before = train.groupby("class_dir").size()
after = clean_train.groupby("class_dir").size()
cmp = pd.DataFrame({"train sebelum": before,
                    "dikeluarkan": before.sub(after, fill_value=0).astype(int),
                    "train bersih": after}).fillna(0).astype(int)
cmp.loc["TOTAL"] = cmp.sum()
print(cmp.to_string())
print(f"\nTotal dikeluarkan dari train: {len(excluded)} (leakage {len(excl)} + duplikat dalam-split {len(dup_excl)} + near-dup lintas split {len(near_excl)})")
print(f"Citra train identik dengan test (sisa): {remaining_leak}")
print(f"Duplikat di dalam train bersih (sisa) : {dups_in_clean}")
print("Ditulis:", CLEAN_CSV.relative_to(PROJECT_DIR).as_posix())

                 train sebelum  dikeluarkan  train bersih
class_dir                                                
1_cumulus                  775            0           775
2_altocumulus              725            0           725
3_cirrus                  1153            0          1153
4_clearsky                2150          187          1963
5_stratocumulus           1846            0          1846
6_cumulonimbus            3003            0          3003
7_mixed                    348            2           346
TOTAL                    10000          189          9811

Total dikeluarkan dari train: 189 (leakage 156 + duplikat dalam-split 1 + near-dup lintas split 32)
Citra train identik dengan test (sisa): 0
Duplikat di dalam train bersih (sisa) : 0
Ditulis: reports/tables/gcd_raw_train_clean_manifest.csv


## 5. Bobot kelas (class imbalance)

Imbalance sedang (rasio train 8,63; `docs/dataset/DATASET_QUALITY_SUMMARY.md` §2.3). Penanganannya di sisi training, bukan data: bobot kelas dihitung dari **train bersih saja** (setelah leakage & duplikat dibuang), **bukan** dari test, dan test **tidak** diseimbangkan.

Rumus inverse-frequency (setara `sklearn` `class_weight='balanced'`): `w_c = N / (K * n_c)`, dengan `N` = total train bersih, `K` = jumlah kelas, `n_c` = jumlah citra kelas `c`. `index` mengikuti urutan kelas untuk `CrossEntropyLoss(weight=...)`; `sampler_weight_per_image` disediakan untuk alternatif `WeightedRandomSampler`.

In [5]:
CLASS_NAMES = ["Cu", "Ac", "Ci", "Cl", "Sc", "Cb", "Mi"]  # urutan folder 1..7, sesuai configs/base.yaml
classes = sorted(clean_train["class_dir"].unique())       # 1_cumulus ... 7_mixed
assert len(classes) == len(CLASS_NAMES), "Jumlah kelas tidak cocok dengan CLASS_NAMES di configs/base.yaml."

counts = clean_train["class_dir"].value_counts().reindex(classes)
N, K = int(counts.sum()), len(classes)
weights = N / (K * counts)        # inverse-frequency ('balanced')
sample_w = 1.0 / counts           # bobot per-citra untuk WeightedRandomSampler

cw = pd.DataFrame({
    "index": range(K),
    "class_dir": classes,
    "class_name": CLASS_NAMES,
    "n_train_clean": counts.astype(int).to_numpy(),
    "weight_inverse_freq": weights.round(4).to_numpy(),
    "sampler_weight_per_image": sample_w.round(6).to_numpy(),
})
CW_CSV = TABLE_DIR / f"{DATASET}_class_weights.csv"
cw.to_csv(CW_CSV, index=False, encoding="utf-8")

print(cw.to_string(index=False))
print(f"\nRasio bobot maks/min: {weights.max() / weights.min():.2f}  (= rasio frekuensi; mayoritas Cb, minoritas Mi)")
print("Dipakai di training lewat configs/base.yaml -> training.class_weights_file (CrossEntropyLoss(weight=...)).")
print("Catatan: dihitung dari TRAIN BERSIH saja; test tidak diseimbangkan.")
print("Ditulis:", CW_CSV.relative_to(PROJECT_DIR).as_posix())

 index       class_dir class_name  n_train_clean  weight_inverse_freq  sampler_weight_per_image
     0       1_cumulus         Cu            775               1.8085                  0.001290
     1   2_altocumulus         Ac            725               1.9332                  0.001379
     2        3_cirrus         Ci           1153               1.2156                  0.000867
     3      4_clearsky         Cl           1963               0.7140                  0.000509
     4 5_stratocumulus         Sc           1846               0.7592                  0.000542
     5  6_cumulonimbus         Cb           3003               0.4667                  0.000333
     6         7_mixed         Mi            346               4.0508                  0.002890



Rasio bobot maks/min: 8.68  (= rasio frekuensi; mayoritas Cb, minoritas Mi)
Dipakai di training lewat configs/base.yaml -> training.class_weights_file (CrossEntropyLoss(weight=...)).
Catatan: dihitung dari TRAIN BERSIH saja; test tidak diseimbangkan.
Ditulis: reports/tables/gcd_raw_class_weights.csv


## 6. Split train / validation

GCD hanya punya train & test; validation dibuat dari **train bersih** (bukan dari test). Pembagian **stratified** per kelas dengan **seed tetap** (42). **Grup near-duplicate di dalam train dijaga tetap satu sisi** (semua di train atau semua di validation) agar tidak muncul leakage train↔val baru. **Test resmi tidak dibagi ulang.** Logika di `scripts/make_split.py`.

In [6]:
from scripts.make_split import grouped_stratified_split, within_train_edges

VAL_FRACTION = 0.15  # bagian validation dari train bersih; sejalan dengan configs/base.yaml split.val

clean_rows = clean_train.to_dict("records")
edges = within_train_edges(TABLE_DIR / f"{DATASET}_eda_near_duplicate_candidates.csv")
val_paths = grouped_stratified_split(clean_rows, edges, VAL_FRACTION, SEED)

split_train = clean_train[~clean_train["rel_path"].isin(val_paths)].assign(split="train").reset_index(drop=True)
split_val = clean_train[clean_train["rel_path"].isin(val_paths)].assign(split="val").reset_index(drop=True)
split_test = man[man["split"] == "test"].reset_index(drop=True)   # test resmi, tidak diubah

for nm, df in [("train", split_train), ("val", split_val), ("test", split_test)]:
    df.to_csv(TABLE_DIR / f"{DATASET}_split_{nm}_manifest.csv", index=False, encoding="utf-8")

# Verifikasi
tr, va = set(split_train["rel_path"]), set(split_val["rel_path"])
assert tr.isdisjoint(va), "train dan val beririsan."
assert tr | va == set(clean_train["rel_path"]), "train+val tidak sama dengan train bersih."
for a, b in edges:                                   # grup near-dup dalam train tidak boleh terpisah
    if a in (tr | va) and b in (tr | va):
        assert (a in tr) == (b in tr), f"pasangan near-dup terpisah: {a} | {b}"
val_leak = int(split_val["pixel_sha256"].isin(set(split_test["pixel_sha256"])).sum())
assert val_leak == 0, f"{val_leak} citra val identik dengan test."

tab = pd.DataFrame({"train": split_train["class_dir"].value_counts(),
                    "val": split_val["class_dir"].value_counts()}).fillna(0).astype(int)
tab["val %"] = (100 * tab["val"] / (tab["train"] + tab["val"])).round(2)
tot_tr, tot_va = int(tab["train"].sum()), int(tab["val"].sum())
tab.loc["TOTAL"] = [tot_tr, tot_va, round(100 * tot_va / (tot_tr + tot_va), 2)]
print(tab.to_string())
print(f"\nSeed: {SEED} | val fraction: {VAL_FRACTION} | grup near-dup dijaga: {len(edges)}")
print(f"Train {len(split_train)} | Val {len(split_val)} | Test {len(split_test)} (tidak diubah)")
print(f"Citra val identik dengan test (sisa): {val_leak}")

                  train     val  val %
class_dir                             
6_cumulonimbus   2553.0   450.0  14.99
4_clearsky       1669.0   294.0  14.98
5_stratocumulus  1568.0   278.0  15.06
3_cirrus          980.0   173.0  15.00
1_cumulus         659.0   116.0  14.97
2_altocumulus     617.0   108.0  14.90
7_mixed           294.0    52.0  15.03
TOTAL            8340.0  1471.0  14.99

Seed: 42 | val fraction: 0.15 | grup near-dup dijaga: 3
Train 8340 | Val 1471 | Test 9000 (tidak diubah)
Citra val identik dengan test (sisa): 0


## 7. Pipeline praproses (resize + normalisasi)

Semua citra 512x512 RGB dan seragam, jadi pipeline cukup di memori saat pemuatan — **citra tidak ditulis ulang** ke `data/processed/`. Urutan: **decode RGB → resize ke `image_size` (224, bilinear) → normalisasi**. Statistik normalisasi dihitung dari **split train final** (8.367 citra; bukan val/test) pada skala 0-1, memakai `channel_mean_std` dari `src/datasets/image_stats.py` atas tabel `gcd_raw_eda_image_stats.csv` (eksak karena semua citra berukuran sama).

In [7]:
from src.datasets.image_stats import channel_mean_std

IMAGE_SIZE = 224  # = configs/base.yaml data.image_size
img_stats = pd.read_csv(TABLE_DIR / f"{DATASET}_eda_image_stats.csv")
train_stats = img_stats[img_stats["rel_path"].isin(set(split_train["rel_path"]))]
assert len(train_stats) == len(split_train), "statistik train tidak lengkap."

mean, std = channel_mean_std(train_stats)   # skala 0-1
norm = pd.DataFrame({
    "channel": ["R", "G", "B"],
    "mean_0_1": mean.round(4),
    "std_0_1": std.round(4),
    "mean_0_255": (mean * 255).round(2),
    "std_0_255": (std * 255).round(2),
})
NORM_CSV = TABLE_DIR / f"{DATASET}_norm_stats.csv"
norm.to_csv(NORM_CSV, index=False, encoding="utf-8")

print(norm.to_string(index=False))
print(f"\nPipeline (di memori): decode RGB -> resize {IMAGE_SIZE} (bilinear) -> normalize(mean, std)")
print("mean =", [round(float(x), 4) for x in mean], "| std =", [round(float(x), 4) for x in std], "(skala 0-1, split train final)")
print("Dipakai lewat configs/base.yaml -> preprocess.normalize; diterapkan di kode dataset/training (torchvision).")
print("Ditulis:", NORM_CSV.relative_to(PROJECT_DIR).as_posix())

channel  mean_0_1  std_0_1  mean_0_255  std_0_255
      R    0.4885   0.1918      124.58      48.92
      G    0.6170   0.1297      157.32      33.07
      B    0.7650   0.1403      195.08      35.77

Pipeline (di memori): decode RGB -> resize 224 (bilinear) -> normalize(mean, std)
mean = [0.4885, 0.617, 0.765] | std = [0.1918, 0.1297, 0.1403] (skala 0-1, split train final)
Dipakai lewat configs/base.yaml -> preprocess.normalize; diterapkan di kode dataset/training (torchvision).
Ditulis: reports/tables/gcd_raw_norm_stats.csv


Catatan penerapan (tahap training, belum ditulis): resize/normalisasi diterapkan saat pemuatan batch, misalnya `transforms.Compose([Resize(224), ToTensor(), Normalize(mean, std)])`. Augmentasi (lihat `configs/base.yaml` `augmentation`) ditambahkan hanya pada split **train**, tidak pada val/test. Nilai mean/std di atas sudah tersimpan di `configs/base.yaml` `preprocess.normalize`.

## 8. Penggabungan dataset (GCD + CCSN_v2)

Opsional: menggabungkan dataset kedua **CCSN_v2** (11 genera WMO) ke skema 7 kelas GCD, lalu menambahkannya ke **train/validation**. **Test tetap test resmi GCD** (tidak ditambah). Pemetaan: contrails (Ct) → `3_cirrus`; dilakukan deduplikasi internal CCSN, pembuangan citra berlabel ambigu, dan pembuangan citra yang identik dengan GCD. Logika di `scripts/combine_datasets.py` (`build_combined`); rincian di `docs/dataset/PENGGABUNGAN_DATASET.md`.

Bagian ini hanya berjalan bila `reports/tables/ccsn_v2_raw_manifest.csv` sudah ada — yaitu setelah `notebooks/01_dataset_analysis.ipynb` dijalankan dengan `data/raw/CCSN_v2/` tersedia.

In [8]:
from scripts.combine_datasets import CCSN_TO_GCD, build_combined

CCSN_MANIFEST = TABLE_DIR / "ccsn_v2_raw_manifest.csv"
if not CCSN_MANIFEST.exists():
    print("Dilewati: ccsn_v2_raw_manifest.csv belum ada. Jalankan notebook 01 untuk CCSN_v2 dulu.")
else:
    splits_c, summary_c = build_combined(TABLE_DIR, PROJECT_DIR / "reports" / "metrics")
    print(f"Pemetaan Ct -> {CCSN_TO_GCD['Ct']} | CCSN dipakai: {summary_c['ccsn_kept']} dari {summary_c['ccsn_raw']}")
    print(f"Dibuang CCSN: {summary_c['ccsn_dedup']}")
    print(f"CCSN identik dengan test GCD (leakage): {summary_c['ccsn_leakage_with_gcd_test']}")

    comp = pd.DataFrame({s: splits_c[s]["class"].value_counts() for s in ["train", "val", "test"]}).fillna(0).astype(int).sort_index()
    comp["Total"] = comp.sum(axis=1)
    comp.loc["TOTAL"] = comp.sum()
    print("\nKomposisi gabungan per kelas:")
    print(comp.to_string())
    print("\nAsal per split:", {s: dict(splits_c[s]["dataset"].value_counts()) for s in ["train", "val"]})
    print("Ditulis: reports/tables/combined_{train,val,test}_manifest.csv, combined_class_weights.csv, combined_class_mapping.csv")

Pemetaan Ct -> 3_cirrus | CCSN dipakai: 2523 dari 2543
Dibuang CCSN: {'dropped_internal_duplicate': 14, 'dropped_ambiguous_label': 6}
CCSN identik dengan test GCD (leakage): 0

Komposisi gabungan per kelas:
                 train   val  test  Total
class                                    
1_cumulus          809   143   750   1702
2_altocumulus     1027   182   750   1959
3_cirrus          1511   267   753   2531
4_clearsky        1669   294  1589   3552
5_stratocumulus   2187   386  1790   4363
6_cumulonimbus    2986   527  2761   6274
7_mixed            294    52   607    953
TOTAL            10483  1851  9000  21334

Asal per split: {'train': {'GCD': 8338, 'CCSN': 2145}, 'val': {'GCD': 1473, 'CCSN': 378}}
Ditulis: reports/tables/combined_{train,val,test}_manifest.csv, combined_class_weights.csv, combined_class_mapping.csv


## 9. Hasil

- Leakage lintas split (piksel identik): `reports/tables/gcd_raw_leakage_exclusion.csv` (156 citra train).
- Near-duplicate lintas split terkonfirmasi (d≤2, tinjauan visual): `reports/tables/gcd_raw_near_dup_crosssplit_exclusion.csv` (32 citra train).
- Duplikat dalam-split (dikeluarkan): `reports/tables/gcd_raw_within_split_duplicate_exclusion.csv` (1 citra train).
- Duplikat test (dicatat, tidak dikeluarkan): `reports/tables/gcd_raw_test_duplicates.csv` (2 citra).
- Manifest train bersih: `reports/tables/gcd_raw_train_clean_manifest.csv` (**9.811 dari 10.000** citra; 189 dibuang).
- Bobot kelas: `reports/tables/gcd_raw_class_weights.csv`; statistik normalisasi: `reports/tables/gcd_raw_norm_stats.csv`.
- Split final GCD (stratified, seed 42): `reports/tables/gcd_raw_split_{train,val,test}_manifest.csv`; ringkasan `reports/metrics/gcd_raw_split_summary.json`.
- **Gabungan GCD+CCSN** (bagian 8): `reports/tables/combined_{train,val,test}_manifest.csv` (train 10.483 / val 1.851 / test 9.000), `combined_class_weights.csv`, `combined_class_mapping.csv`.
- Verifikasi: 0 citra train bersih identik dengan test; 0 duplikat tersisa di dalam train; 0 citra val identik dengan test; grup near-duplicate dalam train tidak terpisah train/val; 0 citra CCSN identik dengan test GCD.

Preprocessing berbasis manifest **selesai**. Tahap training dapat memakai **manifest GCD-only** (`gcd_raw_split_*`) atau **gabungan** (`combined_*`), ditambah `preprocess.normalize` dan `training.imbalance` dari `configs/base.yaml`. Folder `data/raw/` tidak diubah; `data/processed/` tidak ditulis (pipeline di memori).

### Sisa (tahap lain)

- **Training & evaluasi**: tulis dataset/dataloader (terapkan `preprocess`), loss berbobot (`training.imbalance`), dan evaluasi macro-F1 + per kelas (`evaluation`). Di luar cakupan notebook preprocessing ini.